In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from torchvision.datasets import ImageFolder

import jax
import jax.numpy as jnp
from jax import random

# random seed
main_RNG = random.PRNGKey(42)

import flax
from flax import linen as nn
from flax.training import train_state, checkpoints

import optax
from pathlib import Path
print(f"running on: {jax.devices()[0]}")

running on: TFRT_CPU_0


In [13]:
def image_to_numpy(img):
    img = np.array(img, dtype=np.float32)
    if img.max() > 1:
        img = img /255. * 2. - 1.
    return

def jax_to_torch(imgs):
    imgs = jax.device_get(imgs)
    imgs = torch.from_numpy(imgs.astype(np.float32))
    imgs = imgs.permute(0, 3, 1, 2)
    return imgs

def numpy_collate(batch):
    if isinstance(batch[0], np.ndarray):
        return np.stack(batch)
    elif isinstance(batch[0], (tuple, list)):
        transposed = zip(*batch)
        return [numpy_collate(samples) for samples in transposed]
    else:
        return np.array(batch)

data_path = Path(r"C:\Users\bamilosin\Documents\datasets\vision\food vision data\pizza_steak_sushi-10%")

transforms = T.Compose([
    T.ToImage(),
    T.Resize((128,128)),
    T.ToDtype(torch.float)
])

train_dataset = ImageFolder(data_path / "train", transforms)
val_dataset = ImageFolder(data_path / "test", transforms)

batch_size=64
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, collate_fn=numpy_collate)
val_dataloader = DataLoader(val_dataset, batch_size, shuffle=False, collate_fn=numpy_collate)

## autoencoder in jax

In [16]:
class Encoder(nn.Module):
    c_hid: int
    latent_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Conv(features=self.c_hid, kernel_size=(3,3), strides=2)(X)
        x = nn.gelu(x)
        x = nn.Conv(features=self.c_hid, kernel_size=(3,3))(x)
        x = nn.gelu(x)
        x = Conv(features=self.c_hid, kernel_size=(3,3), strides=2)(x)
        x = nn.gelu(x)
        x = Conv(features=self.c_hid, kernel_size=(3,3))(x)
        x = nn.gelu(x)
        x = nn.Conv(features=2*self.c_hid, kernel_size=(3,3), strides=2)(x)
        x - nn.gelu(x)
        x = x.reshape(x.shape[0], -1)
        x = nn.Dense(features=self.latent_dim)(x)
        return x
